In [ ]:
using Distributed
using Plots
using StatsBase
using Random
include("../../../src/helpers/__init__.jl")

using .Helpers.Diffusion: c_next_SOR_sink!, solve_until_tol
using .Helpers.DistributedGIF: distributed_gif


In [ ]:

function create_init_grid(size::Int)
    grid = zeros(size, size)
    grid[:,end] .= 1
    return grid
end

function create_init_sink(size::Int)
    grid = falses(size,size)
    grid[Int64(floor(end/2) +1), 1] = true
    return Matrix(grid)
end


function find_object_neighbors(grid::Matrix{Bool})
    size = length(grid[:,1]) # vertical size
    size_h = length(grid[1,:]) #horizontal size
    
    if size != size_h
        throw("object grid is not square v: $size , h: $size_h " )
        return
    end

    potential_ancors = falses(size, size)
    for i in 1:size
        if any(grid[i,:]) == true

            for j in 1:size
                left = (i == 1) ? size : i - 1 
                right = (i == size) ? 1 : i + 1

                # no periodicity for vertical axis 
                upper = (j == size) ? size : j + 1 
                lower = (j == 1) ? 1 : j - 1

                if grid[i,j] == true # set the neighbors to 1 if not part of obj, 0 otherwise
                    potential_ancors[left, j] = xor( Int(grid[left, j]), 1)
                    potential_ancors[right, j] = xor( Int(grid[right, j]), 1)
                    potential_ancors[i, upper] = xor( Int(grid[i, upper]), 1)
                    potential_ancors[i, lower] = xor( Int(grid[i, lower]), 1)
                end
            end
        end
    end
    return findall(potential_ancors)
end

function pick_neighbor(neighbors::Vector{CartesianIndex{2}}, grid::Matrix{Float64}; eta = 1)
    values = Vector{Tuple{CartesianIndex{2}, Float64}}()
    for neighbor in neighbors
        value = grid[neighbor]
        push!(values, (neighbor, value))
    end
    
    total_gradient = sum(last, values)
    weights = Float64[]
    for i in 1: length(values)
        chance = (max(0, values[i][2])^eta)/(total_gradient^eta)
        push!(weights, chance)
    end
    choice = StatsBase.sample(values, Weights(weights))
    return choice[1]
end

function grow_object_for(iterations; size=100, eta = 1, make_gif = false)
    grid = create_init_grid(size)
    sink = create_init_sink(size)
    if make_gif
        im_list = Matrix{Float64}[]
    end

    for i in 1:iterations
        grid, deltas = solve_until_tol(c_next_SOR_sink!, grid, 10^-4, 100000, 1.91, sink, quiet=true)
        neighbors = find_object_neighbors(sink)
        cords = pick_neighbor(neighbors, grid, eta = eta) 
        sink[cords] = true
        if make_gif
            push_grid = copy(grid)
            push_grid[sink] .= 1
            push!(im_list, push_grid)
        end
    end
    
    if make_gif
        heatmap()
        plots = @distributed (vcat) for i in 1:iterations
            heatmap(im_list[i]')
        end
        gif = distributed_gif(plots, joinpath("plots", "assignment_1_eta=$eta.gif"), width = 600, fps=60)
    else
        grid[sink] .= 1
        heatmap(grid')
        display(current())
    end
end

grow_object_for(600, make_gif=true, eta=1)



In [ ]:
using ProgressMeter


mutable struct GridPoint
    u::Float64
    v::Float64
end

seed = Xoshiro(42);

#TODO add small amount of noise
function create_ass3_grid(size::Int; mini_size::Int = 3 , init_u = 0.5, init_v = 0.25)
    global seed

    if size % 2 == 0 #makes creating the mini square much easier
        throw("size % 2 is not 1, right now size % 2 == 0")  
    end
    grid = [GridPoint(init_u, 0) for _ in 1:size, _ in 1:size]
    
    #weird logic to find and fill small square in the middle 
    mid = Int(ceil(size / 2))
    mid_diference = Int(((mini_size - 1) / 2))
    for p in grid[mid-mid_diference : mid+mid_diference, mid-mid_diference : mid+mid_diference] 
        p.v = init_v
    end

    for p in grid 
        perturb = randn(seed, 2) / 2
        p.v = (p.v + perturb[1] < 0 || p.v + perturb[1] > 1) ? p.v : p.v + perturb[1] 
        p.u = (p.u + perturb[2] < 0 || p.u + perturb[2] > 1) ? p.u : p.u + perturb[2] 
    end
    
    return grid
end

function laplacian(u_minus_1::Float64, u::Float64, u_plus_1::Float64; δ::Float64 = 1.0)
    return (u_plus_1 - 2 * u + u_minus_1) / δ^2
end

function ∂u(x::Int, y::Int, grid::Matrix{GridPoint}; k::Float64=0.0,
    D_u::Float64= 0.16, D_v::Float64= 0.08, f::Float64= 0.035, δy::Float64 = 1.0, δx::Float64 = 1.0)

    size_x, size_y = size(grid)
    @assert size_x == size_y

    #central diff setup with periodic boundaries
    x_minus = (x == 1) ? size_x : x - 1 
    x_plus = (x == size_x) ? 1 : x + 1
    y_minus = (y == 1) ? size_y : y - 1
    y_plus = (y == size_y) ? 1 : y + 1

    u = grid[x,y].u
    u_x_minus = grid[x_minus, y].u
    u_x_plus = grid[x_plus, y].u
    u_y_minus = grid[x, y_minus].u
    u_y_plus = grid[x, y_plus].u

    v = grid[x,y].v

    return D_u * (laplacian(u_x_minus, u, u_x_plus, δ = δx) + laplacian(u_y_minus, u, u_y_plus, δ = δy)) - (u * v^2)  + f * (1-u) 
end

function ∂v(x::Int, y::Int, grid::Matrix{GridPoint}; D_u::Float64= 0.16, 
    D_v::Float64= 0.08, f::Float64= 0.035, k::Float64= 0.065, δy::Float64 = 1.0, δx::Float64 = 1.0 )

    size_x, size_y = size(grid)
    @assert size_x == size_y

    #central diff setup with periodic boundaries
    x_minus = (x == 1) ? size_x : x - 1 
    x_plus = (x == size_x) ? 1 : x + 1
    y_minus = (y == 1) ? size_y : y - 1
    y_plus = (y == size_y) ? 1 : y + 1

    v = grid[x,y].v
    v_x_minus = grid[x_minus, y].v
    v_x_plus = grid[x_plus, y].v
    v_y_minus = grid[x, y_minus].v
    v_y_plus = grid[x, y_plus].v

    u = grid[x,y].u

    return D_v * (laplacian(v_x_minus, v, v_x_plus, δ = δx) + laplacian(v_y_minus, v, v_y_plus, δ = δy)) + (u * v^2) - (f + k) * v
end

function run_gray_scott(iterations::Int, size::Int; δt = 1,  gif = false, kwargs... )
    if gif
        im_list = Matrix{Float64}[]
    end
    
    grid = create_ass3_grid(size)

    @showprogress for i in 1:iterations
        new_grid = Matrix{GridPoint}(undef, size, size)
        
        for x in 1:size
            for y in 1:size
                new_u = δt * ∂u(x, y, grid; kwargs...)
                new_v = δt * ∂v(x, y, grid; kwargs...)
                new_grid[x,y] = GridPoint(grid[x,y].u + new_u, grid[x,y].v + new_v)
            end
        end
        grid = new_grid
        
        if gif
            push!(im_list, getfield.(grid, :u))
        end
    end
    if gif
        heatmap()
        plots = @distributed (vcat) for i in 1:iterations
            heatmap(im_list[i])
            title!("iteration $i")
        end
        gif = distributed_gif(plots, joinpath("plots", "assignment_3_$iterations.mp4"), width = 600, fps=60)
    end

    return grid
end


result = run_gray_scott(5000, 101, gif = false, δt=1.0 ,δx=1.0, δy=1.0, k=0.060, f=0.035)
v_vals = getfield.(result, :u)
heatmap(v_vals)
display(current())

